# M52 — Performance Testing

## AuthentiScore

This notebook measures the runtime performance of the AuthentiScore FastAPI application.

### Endpoints tested

1. `/health`
2. `/scan/validate-url`
3. `/scan/profile-scan`

### Test configuration

- 20 requests per endpoint
- Sequential requests
- Response time measured in milliseconds

### Metrics

- Minimum
- Mean
- Median
- P95
- Maximum
- Success rate

**Important:** The API server must already be running at `http://127.0.0.1:8000`.

In [1]:
import time
import statistics
from pathlib import Path

import requests
import pandas as pd
import numpy as np

In [2]:
API_BASE_URL = "http://127.0.0.1:8000"
REQUESTS_PER_ENDPOINT = 20

print("API Base URL:", API_BASE_URL)
print("Requests per endpoint:", REQUESTS_PER_ENDPOINT)

API Base URL: http://127.0.0.1:8000
Requests per endpoint: 20


In [3]:
try:
    response = requests.get(
        f"{API_BASE_URL}/health",
        timeout=10
    )

    print("Status code:", response.status_code)
    print("Response:", response.json())

    if response.ok:
        print("\n✓ AuthentiScore API is reachable")
    else:
        raise RuntimeError(
            "Health endpoint returned an unsuccessful response."
        )

except requests.exceptions.RequestException as error:
    raise RuntimeError(
        "Could not connect to AuthentiScore backend. "
        "Start Uvicorn at http://127.0.0.1:8000 first."
    ) from error

Status code: 200
Response: {'status': 'healthy'}

✓ AuthentiScore API is reachable


In [4]:
def measure_request(
    method,
    url,
    requests_count=20,
    **kwargs
):
    response_times = []
    status_codes = []
    errors = []

    for i in range(requests_count):
        start = time.perf_counter()

        try:
            response = requests.request(
                method,
                url,
                timeout=30,
                **kwargs
            )

            elapsed_ms = (
                time.perf_counter() - start
            ) * 1000

            response_times.append(elapsed_ms)
            status_codes.append(response.status_code)

        except requests.exceptions.RequestException as error:
            elapsed_ms = (
                time.perf_counter() - start
            ) * 1000

            response_times.append(elapsed_ms)
            status_codes.append(None)
            errors.append(str(error))

    successful = sum(
        200 <= status_code < 300
        for status_code in status_codes
        if status_code is not None
    )

    valid_times = [
        value
        for value, status_code in zip(
            response_times,
            status_codes
        )
        if status_code is not None
    ]

    if not valid_times:
        raise RuntimeError(
            f"No valid responses received from {url}"
        )

    return {
        "response_times": valid_times,
        "status_codes": status_codes,
        "successful": successful,
        "total": requests_count,
        "errors": errors
    }

In [5]:
def calculate_statistics(
    endpoint,
    measurement
):
    times = measurement["response_times"]

    return {
        "endpoint": endpoint,
        "requests": measurement["total"],
        "successful_requests": measurement["successful"],
        "success_rate_percent": (
            measurement["successful"]
            / measurement["total"]
            * 100
        ),
        "min_ms": min(times),
        "mean_ms": statistics.mean(times),
        "median_ms": statistics.median(times),
        "p95_ms": float(np.percentile(times, 95)),
        "max_ms": max(times)
    }

## 1. Health Endpoint

Tests:

```text
GET /health
```

In [6]:
health_result = measure_request(
    "GET",
    f"{API_BASE_URL}/health",
    REQUESTS_PER_ENDPOINT
)

health_stats = calculate_statistics(
    "/health",
    health_result
)

health_stats

{'endpoint': '/health',
 'requests': 20,
 'successful_requests': 20,
 'success_rate_percent': 100.0,
 'min_ms': 1.4955000078771263,
 'mean_ms': 12.829144997522235,
 'median_ms': 15.278449995093979,
 'p95_ms': 16.59828498086427,
 'max_ms': 24.971299979370087}

## 2. URL Validation Endpoint

Tests:

```text
POST /scan/validate-url
```

A supported Instagram profile URL is used only as a test input.
No external profile data is collected by this benchmark.

In [7]:
VALIDATION_URL = (
    f"{API_BASE_URL}/scan/validate-url"
    "?profile_url="
    "https%3A%2F%2Fwww.instagram.com%2Finstagram%2F"
)

validation_result = measure_request(
    "POST",
    VALIDATION_URL,
    REQUESTS_PER_ENDPOINT,
    headers={
        "Content-Type": "application/json"
    }
)

validation_stats = calculate_statistics(
    "/scan/validate-url",
    validation_result
)

validation_stats

{'endpoint': '/scan/validate-url',
 'requests': 20,
 'successful_requests': 20,
 'success_rate_percent': 100.0,
 'min_ms': 1.7868999857455492,
 'mean_ms': 11.202324996702373,
 'median_ms': 13.426649995381013,
 'p95_ms': 21.841489990765695,
 'max_ms': 23.786900012055412}

## 3. Profile Scan Endpoint

Tests:

```text
POST /scan/profile-scan
```

The request uses synthetic/manual profile values so that the benchmark
does not depend on live social-media collection.

In [8]:
PROFILE_PAYLOAD = {
    "platform": "instagram",
    "profile_url": "https://www.instagram.com/performance_test_user/",
    "username": "performance_test_user",
    "followers": 500,
    "following": 250,
    "posts": 100,
    "bio": "Student and technology enthusiast",
    "has_profile_picture": 1,
    "is_private": 0
}

profile_result = measure_request(
    "POST",
    f"{API_BASE_URL}/scan/profile-scan",
    REQUESTS_PER_ENDPOINT,
    headers={
        "Content-Type": "application/json"
    },
    json=PROFILE_PAYLOAD
)

profile_stats = calculate_statistics(
    "/scan/profile-scan",
    profile_result
)

profile_stats

{'endpoint': '/scan/profile-scan',
 'requests': 20,
 'successful_requests': 20,
 'success_rate_percent': 100.0,
 'min_ms': 33.78690002136864,
 'mean_ms': 55.60468500480056,
 'median_ms': 49.995199995464645,
 'p95_ms': 104.26257000654005,
 'max_ms': 105.21959999459796}

In [9]:
results = pd.DataFrame([
    health_stats,
    validation_stats,
    profile_stats
])

results = results.round({
    "success_rate_percent": 2,
    "min_ms": 2,
    "mean_ms": 2,
    "median_ms": 2,
    "p95_ms": 2,
    "max_ms": 2
})

display(results)

,endpoint,requests,successful_requests,success_rate_percent,min_ms,mean_ms,median_ms,p95_ms,max_ms
0,/health,20,20,100.0,1.50,12.83,15.28,16.60,24.97
1,/scan/validate-url,20,20,100.0,1.79,11.20,13.43,21.84,23.79
2,/scan/profile-scan,20,20,100.0,33.79,55.60,50.00,104.26,105.22


In [10]:
print("=" * 70)
print("M52 — PERFORMANCE TESTING RESULTS")
print("=" * 70)

for _, row in results.iterrows():
    print(f"\nEndpoint: {row['endpoint']}")
    print(f"Requests: {int(row['requests'])}")
    print(
        f"Successful: {int(row['successful_requests'])}"
    )
    print(
        f"Success Rate: {row['success_rate_percent']:.2f}%"
    )
    print(
        f"Min: {row['min_ms']:.2f} ms"
    )
    print(
        f"Mean: {row['mean_ms']:.2f} ms"
    )
    print(
        f"Median: {row['median_ms']:.2f} ms"
    )
    print(
        f"P95: {row['p95_ms']:.2f} ms"
    )
    print(
        f"Max: {row['max_ms']:.2f} ms"
    )

M52 — PERFORMANCE TESTING RESULTS

Endpoint: /health
Requests: 20
Successful: 20
Success Rate: 100.00%
Min: 1.50 ms
Mean: 12.83 ms
Median: 15.28 ms
P95: 16.60 ms
Max: 24.97 ms

Endpoint: /scan/validate-url
Requests: 20
Successful: 20
Success Rate: 100.00%
Min: 1.79 ms
Mean: 11.20 ms
Median: 13.43 ms
P95: 21.84 ms
Max: 23.79 ms

Endpoint: /scan/profile-scan
Requests: 20
Successful: 20
Success Rate: 100.00%
Min: 33.79 ms
Mean: 55.60 ms
Median: 50.00 ms
P95: 104.26 ms
Max: 105.22 ms


In [11]:
possible_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent
]

PROJECT_ROOT = None

for root in possible_roots:
    if (
        (root / "data").exists()
        and (root / "backend").exists()
    ):
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate AuthentiScore project root."
    )

PROCESSED_DIR = (
    PROJECT_ROOT /
    "data" /
    "processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_PATH = (
    PROCESSED_DIR /
    "performance_results.csv"
)

results.to_csv(
    OUTPUT_PATH,
    index=False
)

print("✓ Performance results saved:")
print(OUTPUT_PATH)

✓ Performance results saved:
c:\Users\Acer\Documents\AuthentiScore\data\processed\performance_results.csv


## M52 Interpretation

Use the measured values above in the project report.

Do **not** claim that an endpoint is "fast", "slow", or production-ready
based only on these numbers. The measurements are environment-dependent
and were collected from the local development API.

The results should be reported as an application performance benchmark
under the tested local configuration.

In [12]:
print("=" * 70)
print("M52 — PERFORMANCE TESTING COMPLETE")
print("=" * 70)

print("\nSaved output:")
print(OUTPUT_PATH)

print("\nEndpoints tested:")
print("1. /health")
print("2. /scan/validate-url")
print("3. /scan/profile-scan")

print("\nRequests per endpoint:", REQUESTS_PER_ENDPOINT)
print("\nM52 completed.")

M52 — PERFORMANCE TESTING COMPLETE

Saved output:
c:\Users\Acer\Documents\AuthentiScore\data\processed\performance_results.csv

Endpoints tested:
1. /health
2. /scan/validate-url
3. /scan/profile-scan

Requests per endpoint: 20

M52 completed.
